## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [ ]:
# The imports + FREE-first bootstrap for the OpenAI Agents SDK
#
# ====== CRITICAL TECH NOTE ======
# The `agents` SDK uses the NEW Responses API (POST /responses), NOT the older
# /chat/completions endpoint. That means MOST "OpenAI-compatible" FREE providers
# do NOT work with Agent(...)/Runner.run() — only providers that implement the
# /responses endpoint will avoid HTTP 404. Currently confirmed working:
#   1. Google Gemini NATIVE   (full /responses API beta)
#   2. OpenRouter             (bridges /responses → internal free model)
#   3. OpenAI native          (paid)
# Providers with ONLY /chat/completions (Groq, Ollama, DeepSeek, xAI, Anthropic)
# will ALL throw HTTP 404 on Runner.run(). We therefore put them in a SKIP
# section below and don't auto-select them.
# =================================

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
from agents.run_config import RunConfig

load_dotenv(override=True)

# ---- Kill the noisy 401 trace spam ONCE AND FOR ALL ------------------------
# We use THREE independent knobs together because older agents SDK versions
# don't always honor every env var the same way:
os.environ["AGENTS_TRACING_ENABLED"]    = "false"    # agents SDK own switch
os.environ["OPENAI_TRACING_ENABLED"]    = "false"    # underlying openai SDK switch
os.environ["AGENTS_TRACE_WRITER"]       = "local"    # if local writer used -> writes to disk (never network)
# Also set a default RunConfig that disables tracing programmatically.
# Pass this as `run_config=DEFAULT_RUN_CONFIG` on all Runner.run() calls.
DEFAULT_RUN_CONFIG = RunConfig(tracing_disabled=True, trace_include_sensitive_data=False)

# ---------- Provider detection in /responses-SUPPORT order ----------------
_gm_key  = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")   # Supports /responses NATIVE
_or_key  = os.getenv("OPENROUTER_API_KEY")                               # Supports /responses ✅
_oa_key  = os.getenv("OPENAI_API_KEY")                                  # Supports /responses (paid)
_gr_key  = os.getenv("GROQ_API_KEY")                                    # ⚠️ NO /responses → would 404

# ================ DECIDE PROVIDER ================
# Priority:  OpenRouter FREE (full /responses support)
#            -> Gemini NATIVE (full /responses support)
#            -> OpenAI PAID (last resort, costs money)
_provider_label = None

if _or_key:
    # --- 1. OpenRouter FREE  (best FREE option, /responses works natively on OR as of 2026)
    _provider_label = "OpenRouter FREE — /responses endpoint works on OR"
    _api_key = _or_key
    _base_url = "https://openrouter.ai/api/v1"
    # Model: a free, well-used model. OR will internally route requests.
    MODEL_NAME = "nvidia/nemotron-3-ultra-550b-a55b:free"
    # Fallback chain of FREE models on OpenRouter for resilience.
    # Last entry is `openrouter/free` which is a literal auto-router that can never 404.
    OPENROUTER_FREE_FALLBACKS = [
        ("nvidia/nemotron-3-ultra-550b-a55b:free",  "Nemotron 3 Ultra 550B FREE"),
        ("poolside/laguna-s-2.1:free",              "Laguna S 2.1 FREE (coding)"),
        ("meta-llama/llama-3.3-70b-instruct:free",  "Llama 3.3 70B FREE"),
        ("deepseek/deepseek-r1:free",                "DeepSeek R1 FREE (reasoning)"),
        ("inclusionai/ling-3-0-flash:free",         "Ling 3.0 Flash FREE"),
        ("qwen/qwen3-235b-a22b:free",               "Qwen3 235B FREE"),
        ("openrouter/free",                          "OpenRouter auto-router FREE (cannot 404)"),
    ]
    # Tell the Agents SDK: use OpenAI-compatible /responses mode (OR is openai-compat)
    os.environ.pop("AGENTS_MODEL_PROVIDER", None)

elif _gm_key:
    # --- 2. Google Gemini NATIVE  (full Responses API beta support)
    _provider_label = "Google Gemini NATIVE — /responses endpoint works natively"
    _api_key = _gm_key
    # Do NOT set OPENAI_BASE_URL to the openai-compat endpoint — that doesn't
    # support /responses. Instead, the agents SDK has a GEMINI provider.
    # We tell it to use the Gemini provider explicitly:
    os.environ["AGENTS_MODEL_PROVIDER"] = "gemini"
    # The agents SDK reads Gemini key from AGENTS_GEMINI_API_KEY first
    os.environ["AGENTS_GEMINI_API_KEY"] = _gm_key
    # Still set OPENAI_API_KEY as a generic LLM key (sanity default)
    os.environ["OPENAI_API_KEY"] = _gm_key
    # Gemini stable, long-lived model on free tier
    MODEL_NAME = "gemini-2.5-flash"
    # Remove an openai-compat base_url if any was set earlier
    os.environ.pop("OPENAI_BASE_URL", None)
    os.environ.pop("AGENTS_OPENAI_BASE_URL", None)
    _base_url = None

elif _oa_key:
    # --- 3. OpenAI PAID (last resort only)
    _provider_label = "OpenAI PAID — full /responses native (last resort, costs money)"
    _api_key = _oa_key
    _base_url = None
    MODEL_NAME = "gpt-5.4-mini"
    os.environ.pop("AGENTS_MODEL_PROVIDER", None)

else:
    # --- None of the /responses-compatible providers are configured ---
    # Fall back to Ollama-like warning (Ollama doesn't do /responses either,
    # but at least we don't silently 404). We set a clear "missing config" provider
    # label and a warning MODEL_NAME that will show up in errors.
    _provider_label = (
        "WARNING: None of the /responses-compatible providers (OpenRouter / Gemini / OpenAI) "
        "have keys set. The agents SDK REQUIRES a /responses endpoint. "
        "Groq/Ollama/DeepSeek/xAI CANNOT be used with Runner.run() because they "
        "only implement /chat/completions — they will always return HTTP 404. "
        "Set either GEMINI_API_KEY or OPENROUTER_API_KEY in your .env for a FREE fix."
    )
    _api_key = _gr_key or "local-no-key"
    _base_url = None
    MODEL_NAME = "gemini-2.5-flash"   # Suggested to the user
    os.environ.pop("AGENTS_MODEL_PROVIDER", None)

# ========= Apply provider config to both generic + agents env vars =========
if _provider_label and _base_url:
    # OpenAI-compatible mode (OpenRouter / OpenAI native PAID)
    os.environ["OPENAI_API_KEY"] = _api_key
    os.environ["OPENAI_BASE_URL"] = _base_url
    os.environ["AGENTS_OPENAI_API_KEY"] = _api_key
    os.environ["AGENTS_OPENAI_BASE_URL"] = _base_url
elif _provider_label and _base_url is None and "AGENTS_MODEL_PROVIDER" in os.environ:
    # Native Gemini mode (AGENTS_MODEL_PROVIDER=gemini set above; no openai-compat url)
    os.environ.setdefault("OPENAI_API_KEY", _api_key)
    os.environ["AGENTS_OPENAI_API_KEY"] = _api_key
elif _provider_label and _base_url is None and _oa_key:
    # OpenAI native PAID
    os.environ["OPENAI_API_KEY"] = _api_key
    os.environ["AGENTS_OPENAI_API_KEY"] = _api_key
    os.environ.pop("OPENAI_BASE_URL", None)
    os.environ.pop("AGENTS_OPENAI_BASE_URL", None)

# ========= Status dump =========
print("=" * 68)
print("BOOTSTRAP OK — Agents SDK configured with /responses-compatible provider")
print("=" * 68)
print("Provider : {}".format(_provider_label))
if _base_url:
    print("Endpoint : {}".format(_base_url))
if os.environ.get("AGENTS_MODEL_PROVIDER"):
    print("Mode     : AGENTS_MODEL_PROVIDER = {}".format(os.environ["AGENTS_MODEL_PROVIDER"]))
print("Model    : {}".format(MODEL_NAME))
print("Tracing  : DISABLED (AGENTS_TRACING_ENABLED=false + DEFAULT_RUN_CONFIG.tracing_disabled=True)")
if _gr_key:
    print()
    print("ℹ️  Note: You also have GROQ_API_KEY set in .env, but Groq does NOT")
    print("   support the /responses endpoint used by the agents SDK. Using it")
    print("   would cause HTTP 404 on every Runner.run() call. We are using")
    print("   {} instead — it actually works with this SDK.".format(
        _provider_label.split('—')[0].strip() if '—' in _provider_label else _provider_label
    ))
print()
print("Usage:")
print("   Agent(name=..., instructions=..., model=MODEL_NAME, ...)")
print("   await Runner.run(agent, input, run_config=DEFAULT_RUN_CONFIG)   ← tracing 401s gone")
print()


## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [ ]:

# Make an agent with name, instructions, model
# Uses the FREE provider MODEL_NAME configured in the bootstrap cell above.

agent = Agent(name="Jokester", instructions="You are a joke teller", model=MODEL_NAME)


In [ ]:
# Run the joke with Runner.run(agent, prompt, run_config=DEFAULT_RUN_CONFIG)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents", run_config=DEFAULT_RUN_CONFIG)


In [ ]:
# Here is the final output

print(result.final_output)

In [ ]:
# Here is the detail of the LLM calls

result.to_input_list()

## Adding Observability with a trace

In [ ]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents", run_config=DEFAULT_RUN_CONFIG)
print(result.final_output)

## Now go and look at the trace

https://platform.openai.com/traces

In [ ]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

## Part 2: Adding a tool

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("HEY!!")

In [ ]:
push

In [ ]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [ ]:
push_tool

In [ ]:
push_tool.description

In [ ]:

# FREE-first: use the configured MODEL_NAME instead of a hardcoded paid model

notifier = Agent(name="Notifier", model=MODEL_NAME, instructions="You notify the user upon request", tools=[push_tool])


In [ ]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here", run_config=DEFAULT_RUN_CONFIG)

print(result.final_output)


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [ ]:
# FREE-first agent for memory demos (used by both manual-history and SQLiteSession runs)

agent = Agent(name="Assistant", model=MODEL_NAME)


In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.", run_config=DEFAULT_RUN_CONFIG)
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?", run_config=DEFAULT_RUN_CONFIG)
print(response.final_output)

## Memory approach 1 - just manually pass in the list of dicts

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.", run_config=DEFAULT_RUN_CONFIG)
print(response.final_output)

In [ ]:
response.to_input_list()

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

In [ ]:
response = await Runner.run(agent, next_input, run_config=DEFAULT_RUN_CONFIG)
print(response.final_output)

## Another approach - use OpenAI Agents SDK built in SQLLite session

In [ ]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session, run_config=DEFAULT_RUN_CONFIG)
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?", session=session, run_config=DEFAULT_RUN_CONFIG)
print(response.final_output)

# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>